This section reads from the Delta raw table as a stream, parses the `body_str` JSON payload into explicit columns, keeps the raw `ingested_at` timestamp, and writes the result to `dev.bronze.ttc_alerts_bronze`.

In [0]:
from pyspark.sql import functions as F, types as T

source_table = "dev.raw.ttc_alerts_raw"
target_table = "dev.bronze.ttc_alerts_bronze"
checkpoint_path = "/Volumes/dev/bronze/checkpoints/ttc_alerts_bronze"

ttc_alerts_schema = T.StructType([
    T.StructField("operation", T.StringType(), True),
    T.StructField("id", T.StringType(), True),
    T.StructField("data", T.StructType([
        T.StructField("alert_id", T.StringType(), True),
        T.StructField("route_id", T.StringType(), True),
        T.StructField("stop_id", T.StringType(), True),
        T.StructField("alert_message", T.StringType(), True),
        T.StructField("language", T.StringType(), True),
        T.StructField("cause", T.StringType(), True),
        T.StructField("effect", T.StringType(), True),
        T.StructField("start_time", T.LongType(), True),
        T.StructField("end_time", T.LongType(), True),
        T.StructField("id", T.StringType(), True)
    ]), True)
])

In [0]:
raw_stream = spark.readStream.table(source_table)

bronze_stream = (
    raw_stream
    .select(
        F.from_json(F.col("body_str"), ttc_alerts_schema).alias("payload"),
        F.col("ingested_at"),
    )
    .select(
        F.col("payload.operation").alias("operation"),
        F.col("payload.id").alias("event_id"),
        F.col("payload.data.alert_id").alias("alert_id"),
        F.col("payload.data.route_id").alias("route_id"),
        F.col("payload.data.stop_id").alias("stop_id"),
        F.col("payload.data.alert_message").alias("message"),
        F.col("payload.data.language").alias("language"),
        F.col("payload.data.cause").alias("cause"),
        F.col("payload.data.effect").alias("effect"),
        F.to_timestamp(F.from_unixtime(F.col("payload.data.start_time"))).alias("start_time"),
        F.to_timestamp(F.from_unixtime(F.col("payload.data.end_time"))).alias("end_time"),
        F.col("ingested_at"),
    )
)

In [0]:
query = (
    bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_path)
    .toTable(target_table)
)